# Introduction - Pricing Engine

This notebook validates the `PricingEngine` interface.

# Import libraries

In [15]:
import sys
from pathlib import Path
import logging
import numpy as np
import time

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Setup logging to see what the engine is doing
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

from src.pricing_engine import PricingEngine
from src.models.ssvi import ssvi_iv

# Create engine

In [16]:
engine = PricingEngine(symbol="SPY", r=0.045, q=0.012, max_expiries=4, cache=False)

# Calibrate (synchronous, to avoid background thread noise for testing)

In [17]:
print(f"Is calibrated? {engine.is_calibrated()}")
print("Starting manual calibration...")
engine._calibrate()
print("Calibration complete.")
print(f"Is calibrated? {engine.is_calibrated()}")

2026-07-08 17:23:37,145 - src.pricing_engine - INFO - Calibrating surface for SPY...


Is calibrated? False
Starting manual calibration...
Calibration complete.
Is calibrated? True


# Inspect the parameters after calibration

In [18]:
svi_params = engine.get_svi_params()
ssvi_params = engine.get_ssvi_params()
sabr_params = engine.get_sabr_params()

print("\n--- SSVI Parameters ---")
print(ssvi_params)

print("\n--- SVI Slices ---")
for exp, data in svi_params.items():
    print(f"  {exp}: theta={data['theta']:.6f}, points={len(data['strikes'])}")

print("\n--- SABR Expiries ---")
print(list(sabr_params.keys()))


--- SSVI Parameters ---
{'rho': np.float64(-0.3669408929506024), 'eta': np.float64(0.642921035662082), 'gamma': 0.5}

--- SVI Slices ---
  2026-07-08: theta=0.000040, points=31
  2026-07-09: theta=0.000096, points=60
  2026-07-10: theta=0.000179, points=103
  2026-07-13: theta=0.001693, points=88

--- SABR Expiries ---
['2026-07-08', '2026-07-09', '2026-07-10', '2026-07-13']


# Get Spot and Forward

In [19]:
spot = engine.get_spot()
print(f"Spot: {spot:.2f}")
expiry = list(svi_params.keys())[0]
forward = engine.get_forward(expiry)
print(f"Forward for {expiry}: {forward:.2f}")

Spot: 740.50
Forward for 2026-07-08: 740.51


# Test background threading

In [20]:
print("--- Testing Background Thread ---")
engine.start_background_calibration(interval_ms=1)

time.sleep(3)  # Let it calibrate once in the background
print(f"Is background running? {engine.is_background_running()}")
engine.stop_background_calibration()
print("Stopped background thread.")

2026-07-08 17:23:39,362 - src.pricing_engine - INFO - Starting initial calibration...
2026-07-08 17:23:39,363 - src.pricing_engine - INFO - Calibrating surface for SPY...


--- Testing Background Thread ---


2026-07-08 17:23:41,147 - src.pricing_engine - INFO - Initial calibration completed successfully.
2026-07-08 17:23:41,150 - src.pricing_engine - INFO - Background calibration thread started (interval=1ms).
2026-07-08 17:23:41,151 - src.pricing_engine - INFO - Calibrating surface for SPY...
2026-07-08 17:23:42,858 - src.pricing_engine - INFO - Background recalibration completed successfully.
2026-07-08 17:23:42,862 - src.pricing_engine - INFO - Calibrating surface for SPY...
2026-07-08 17:23:44,187 - src.pricing_engine - INFO - Stopping background calibration thread...
2026-07-08 17:23:44,323 - src.pricing_engine - INFO - Background recalibration completed successfully.
2026-07-08 17:23:44,326 - src.pricing_engine - INFO - Background calibration thread stopped.


Is background running? True
Stopped background thread.


# Plots

## Libraries for plots

In [21]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Data for plotting

In [22]:
# Build a dense strike grid for each expiry
strike_range = np.linspace(
    min([min(data['strikes']) for data in svi_params.values()]),
    max([max(data['strikes']) for data in svi_params.values()]),
    40
)

expiries = sorted(svi_params.keys())
spot = engine.get_spot()

# Pre-compute per-expiry data
plot_data = []
for exp in expiries:
    data = svi_params[exp]
    T = data["T"]
    forward = data["forward"]
    strikes = data["strikes"]
    iv_market = data["ivs"]
    theta = data["theta"]
    svi_a, svi_b, svi_rho, svi_m, svi_sigma = data["svi_params"]
    
    # SSVI IV on the dense grid
    k_grid = np.log(strike_range / forward)
    iv_ssvi = ssvi_iv(k_grid, theta, ssvi_params["rho"], ssvi_params["eta"], T, ssvi_params["gamma"])
    
    # SABR IV on the dense grid
    sabr = sabr_params.get(exp)
    if sabr is not None:
        # SABR needs forward, alpha, beta, rho, nu
        # We already have forward. Use the stored SABR params.
        from src.models.sabr import sabr_iv
        iv_sabr = []
        for k in k_grid:
            # Need to convert log-moneyness k to strike
            strike = forward * np.exp(k)
            iv = sabr_iv(
                f=forward,
                K=strike,
                T=T,
                alpha=sabr["alpha"],
                beta=sabr["beta"],
                rho=sabr["rho"],
                nu=sabr["nu"]
            )
            iv_sabr.append(iv)
        iv_sabr = np.array(iv_sabr)
    else:
        iv_sabr = np.full_like(strike_range, np.nan)
    
    plot_data.append({
        "expiry": exp,
        "T": T,
        "forward": forward,
        "strikes": strikes,
        "iv_market": iv_market,
        "strike_range": strike_range,
        "iv_ssvi": iv_ssvi,
        "iv_sabr": iv_sabr,
        "theta": theta,
        "svi_params": data["svi_params"],
    })

## Volatility Smiles (Market IV vs SSVI vs SABR)

In [23]:
num_slices = len(plot_data)
fig_smiles = make_subplots(
    rows=1, cols=num_slices,
    shared_yaxes=True,
    horizontal_spacing=0.08,
    subplot_titles=[f"{d['expiry']} (T={d['T']:.3f})" for d in plot_data]
)

for i, d in enumerate(plot_data):
    # Market IV (from SVI slice)
    fig_smiles.add_trace(
        go.Scatter(
            x=d["strikes"],
            y=d["iv_market"],
            mode='markers',
            marker=dict(color='blue', size=6),
            name='Market IV' if i == 0 else None,
            legendgroup='Market IV',
            showlegend=(i == 0)
        ),
        row=1, col=i+1
    )
    
    # SSVI Fit (smooth line)
    fig_smiles.add_trace(
        go.Scatter(
            x=d["strike_range"],
            y=d["iv_ssvi"],
            mode='lines',
            line=dict(color='red', width=2),
            name='SSVI' if i == 0 else None,
            legendgroup='SSVI',
            showlegend=(i == 0)
        ),
        row=1, col=i+1
    )
    
    # SABR Fit (dashed line)
    fig_smiles.add_trace(
        go.Scatter(
            x=d["strike_range"],
            y=d["iv_sabr"],
            mode='lines',
            line=dict(color='green', width=2, dash='dot'),
            name='SABR' if i == 0 else None,
            legendgroup='SABR',
            showlegend=(i == 0)
        ),
        row=1, col=i+1
    )
    
    fig_smiles.update_xaxes(title_text="Strike", row=1, col=i+1)

fig_smiles.update_yaxes(title_text="Implied Volatility", row=1, col=1)
fig_smiles.update_layout(
    title_text=f"Smile Fits: Market IV vs SSVI vs SABR (Spot={spot:.2f})",
    height=500,
    width=1200,
    hovermode='x unified'
)
fig_smiles.show()

## Volatility Surface from SSVI

In [24]:
# Use the dense grid to build the full surface
T_range = np.array([d["T"] for d in plot_data])
strike_grid_surface = strike_range  # Use the same dense grid

IV_matrix = []
for d in plot_data:
    # SSVI IV on the dense grid (already computed)
    iv_grid = d["iv_ssvi"]
    IV_matrix.append(iv_grid)

IV_matrix = np.array(IV_matrix)

fig_surface = go.Figure(data=[
    go.Surface(
        z=IV_matrix,
        x=strike_grid_surface,
        y=T_range * 365,  # Convert to days for readability
        colorscale='Viridis',
        hovertemplate='Strike: %{x:.1f}<br>DTE: %{y:.0f}<br>IV: %{z:.2%}<extra></extra>'
    )
])

fig_surface.update_layout(
    title_text=f"SSVI Volatility Surface (rho={ssvi_params['rho']:.3f}, eta={ssvi_params['eta']:.3f})",
    scene=dict(
        xaxis_title='Strike',
        yaxis_title='Days to Expiry',
        zaxis_title='Implied Volatility',
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
    ),
    height=600,
    width=900
)
fig_surface.show()

## ATM Term Structure (ATM IV as a function of expiry)

In [25]:
atm_ivs = []
dtes = []
for d in plot_data:
    atm_iv = engine.get_atm_iv(d["expiry"])
    atm_ivs.append(atm_iv)
    dte = d["T"] * 365
    dtes.append(dte)

fig_term = go.Figure()
fig_term.add_trace(
    go.Scatter(
        x=dtes,
        y=atm_ivs,
        mode='lines+markers',
        marker=dict(size=10),
        line=dict(width=3),
        name='ATM IV'
    )
)
fig_term.update_layout(
    title="ATM Implied Volatility Term Structure",
    xaxis_title="Days to Expiry",
    yaxis_title="ATM Implied Volatility",
    height=400,
    width=600
)
fig_term.show()

# SABR Dynamic Re-pricing Demo

In [27]:
if plot_data and sabr_params:
    expiry = plot_data[0]["expiry"]
    sabr = sabr_params.get(expiry)
    if sabr is not None:
        print("\n--- SABR Dynamic Re-pricing Demo ---")
        T = plot_data[0]["T"]
        forward_base = plot_data[0]["forward"]
        strikes = plot_data[0]["strikes"]
        
        # Base SABR IV
        iv_base = []
        for K in strikes:
            iv = sabr_iv(forward_base, K, T, sabr["alpha"], sabr["beta"], sabr["rho"], sabr["nu"])
            iv_base.append(iv)
        iv_base = np.array(iv_base)
        
        # Shift spot by +1% and -1%
        shift = 0.01
        forward_up = forward_base * (1 + shift)
        forward_down = forward_base * (1 - shift)
        
        iv_up = []
        iv_down = []
        for K in strikes:
            iv_up.append(sabr_iv(forward_up, K, T, sabr["alpha"], sabr["beta"], sabr["rho"], sabr["nu"]))
            iv_down.append(sabr_iv(forward_down, K, T, sabr["alpha"], sabr["beta"], sabr["rho"], sabr["nu"]))
        iv_up = np.array(iv_up)
        iv_down = np.array(iv_down)
        
        fig_dynamic = go.Figure()
        fig_dynamic.add_trace(go.Scatter(x=strikes, y=iv_base, mode='markers', name=f'Base F={forward_base:.2f}', line=dict(width=3)))
        fig_dynamic.add_trace(go.Scatter(x=strikes, y=iv_up, mode='markers', name=f'Spot +1% (F={forward_up:.2f})', line=dict(dash='dot', width=2)))
        fig_dynamic.add_trace(go.Scatter(x=strikes, y=iv_down, mode='markers', name=f'Spot -1% (F={forward_down:.2f})', line=dict(dash='dot', width=2)))
        
        fig_dynamic.update_layout(
            title=f"SABR Dynamic Re-pricing: Spot Shift ±1% for {expiry}",
            xaxis_title="Strike",
            yaxis_title="Implied Volatility",
            height=450,
            width=800
        )
        fig_dynamic.show()
    else:
        print("SABR not available for the first expiry.")
else:
    print("No plot data available for dynamic demo.")


--- SABR Dynamic Re-pricing Demo ---
